# Lewisham · `description` (web: "Description")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly. Lewisham also serves as a check on what we found for Brent.

Lewisham only (golden rule): every query carries the location-based Lewisham filter.
Data comes from the **live API** (the bot's only source), so counts change as the
Datahub is updated.

Workflow: add a step here → discuss the findings → only after agreement update
`semantic/lewisham/` and `knowledge/`.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def lewisham_filter() -> dict:
    # Location-based scope: borough is Lewisham, or borough is missing and Lewisham decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Lewisham"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Lewisham"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `description` for every Lewisham application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": lewisham_filter(), "_source": ["id", "description"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "description"])
print("Lewisham applications:", len(df))

Lewisham applications: 29021


In [3]:
value = df["description"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,1173,4.04
empty or blank,0,0.00
filled,27848,95.96
total,29021,100.00


### Findings · Step 1

Live API, 2026-11-11, 29,021 Lewisham records:

| | Count | Share |
|---|---|---|
| null | 1,173 | 4.04% |
| empty or blank | 0 | 0% |
| filled | 27,848 | 95.96% |

- **1,173 are null; no empty strings.** The 1,151 empty records have no description by
  definition (dropped at load), so 22 others have none. Some are PRE/ records seen
  before (status notebook, Step 3); the 22 are not listed yet.

## Step 2 · Records with no description that are not empty records

Records with no `description` but a status, decision or decision_date (so not dropped by
`drop_empty_records`), with their type, outcome and source system.

In [4]:
FIELDS = ["id", "lpa_app_no", "application_type", "status", "decision", "decision_date", "valid_date",
          "bo_system", "site_name", "url_planning_app"]
body = {"size": 100, "_source": FIELDS, "sort": [{"id": "asc"}],
        "query": {"bool": {"filter": [lewisham_filter()],
                           "must_not": [{"exists": {"field": "description"}}],
                           "should": [{"exists": {"field": "status.raw"}}, {"exists": {"field": "decision.raw"}},
                                      {"exists": {"field": "decision_date"}}],
                           "minimum_should_match": 1}}}
nodesc = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
nodesc["url"] = nodesc.pop("url_planning_app").notna()
nodesc["site_name"] = nodesc["site_name"].str.slice(0, 40)
nodesc["prefix"] = nodesc["lpa_app_no"].str.extract(r"^([A-Za-z]*)", expand=False)
print(len(nodesc), "records")
print(nodesc[["prefix", "bo_system"]].value_counts().to_string())
print()
print(nodesc.drop(columns=["prefix", "lpa_app_no"]).to_string(index=False))

22 records
prefix  bo_system      
DC      CSV Data Import    15
PRE     CSV Data Import     7

                    id application_type                                  status                                decision decision_date valid_date       bo_system                                site_name  url
 Lewisham-DC_19_110608   Prior Approval                                 Refused                                 Refused    25/02/2019 22/01/2019 CSV Data Import      105 DOWANHILL ROAD, LONDON, SE6 1BX True
 Lewisham-DC_19_114164        All Other     No Objection to Proposal (OBS only)     No Objection to Proposal (OBS only)    08/10/2019 08/10/2019 CSV Data Import CRYSTAL PALACE PARK, THICKET ROAD, LONDO True
 Lewisham-DC_19_114630        All Other     No Objection to Proposal (OBS only)     No Objection to Proposal (OBS only)    19/11/2019 18/11/2019 CSV Data Import      73 HILLCREST ROAD, BROMLEY, BR1 4SE True
 Lewisham-DC_19_114677        All Other                          Comment Iss

### Findings · Step 2

22 records, all CSV Data Import with a URL: 15 DC/, 7 PRE/.

- **Most are opinion-type decisions:** No Objection (OBS only) 6, Comment Issued 7,
  Objection Raised (OBS only) 2, Closed 2, plus Withdrawn 2, Refused 1, Approved 1, pending 1.
- **Many site addresses are outside Lewisham** (from `site_name`, not checked further):
  - Bromley: 73 Hillcrest Road, **Bromley BR1**; 8 Lydstep Road, **Chislehurst BR7**;
    94 Crystal Palace Park Road, **Bromley**; Clifford House, Calverley Close, Beckenham;
    the NatWest sports ground, Copers Cope (Beckenham).
  - Southwark: 19 Bush Road, London **(LB Southwark)**; Canada Water Master Plan; Tustin
    Estate; Borough Triangle, Newington.
  - Greenwich: land west of Kidbrooke Park Road.
  - Not placed: Crystal Palace Park, Thames Tideway Tunnel at Earl Pumping Station.
- **So the opinion-type values on DC/ records look like Lewisham's observations on
  neighbouring boroughs' applications** ("OBS" = observations?), consistent with
  DC/26/142577 ("Observations on a proposed development within … Bromley"). Not
  confirmed, and not counted beyond these 22.
- **Golden rule:** these records have borough Lewisham but sites elsewhere. This is the
  open question already with the user, now with more examples.

## Step 3 · Type check and most common descriptions

The Python type of every value in `df` (Step 1). Then the most common descriptions (exact
text, after trimming), their lengths, and placeholder-like texts (very short, or "No
description …").

In [5]:
types = df["description"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
print("Types:", types.value_counts().to_dict())

text = df["description"].dropna().str.strip()
lengths = text.str.len()
print("Length: median", int(lengths.median()), "| shortest", int(lengths.min()), "| longest", int(lengths.max()),
      "| under 10 characters:", int((lengths < 10).sum()))
print()
print("Most common descriptions:")
pd.set_option("display.max_colwidth", 100)
print(text.value_counts().head(15).to_string())
print()
print("Under 10 characters:", text[lengths < 10].value_counts().to_dict())
print("Starting 'No description':", text[text.str.lower().str.startswith("no description")].value_counts().to_dict())

Types: {'str': 27848, 'NoneType': 1173}
Length: median 199 | shortest 1 | longest 4598 | under 10 characters: 3

Most common descriptions:
description
No description provided by LDMS                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [6]:
# Where the placeholder and the very short texts come from.
FIELDS = ["id", "bo_system", "status", "decision", "decision_date", "application_type", "description"]
body = {"size": 300, "_source": FIELDS, "query": {"bool": {"filter": [lewisham_filter(),
        {"match_phrase": {"description": "No description provided by LDMS"}}]}}}
ldms = pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]], columns=FIELDS)
ldms = ldms[ldms["description"].str.strip() == "No description provided by LDMS"]
years = pd.to_datetime(ldms["decision_date"], format="%d/%m/%Y", errors="coerce").dt.year
print("'No description provided by LDMS':", len(ldms), "| bo_system:", ldms["bo_system"].value_counts().to_dict(),
      "| decision years:", int(years.min()), "-", int(years.max()))
print("decisions:", ldms["decision"].fillna("(none)").value_counts().to_dict())
print()
short_ids = df.loc[df["description"].fillna("").str.strip().isin(["FOOB", "V"]), "id"].tolist()
body = {"size": 5, "_source": FIELDS + ["lpa_app_no", "site_name"], "query": {"bool": {"filter": [lewisham_filter(), {"terms": {"id": short_ids}}]}}}
pd.DataFrame([h["_source"] for h in search(body)["hits"]["hits"]]).T

'No description provided by LDMS': 195 | bo_system: {'FME Legacy Data Import': 195} | decision years: 1987 - 1993
decisions: {'Approved': 193, '(none)': 1, 'Refused': 1}



,0,1
bo_system,CSV Data Import,CSV Data Import
site_name,"120 FLEET STREET, LONDON, EC4A 2BE","31 HATCHAM PARK ROAD, LONDON, SE14 5QE"
decision_date,20/07/2021,20/01/2023
lpa_app_no,DC/21/122666,DC/22/129197
decision,Closed,Approved
application_type,All Other,All Other
description,FOOB,V
id,Lewisham-DC_21_122666,Lewisham-DC_22_129197
status,Closed,Approved


### Findings · Step 3

- **Strings or null only** (27,848 strings, 1,173 null); no numbers.
- **Free text, rarely repeated:** median 199 characters (1 to 4,598); apart from the
  placeholder below, no description is used more than 5 times.
- **Placeholder: "No description provided by LDMS" (195).** All are legacy imports
  (FME Legacy Data Import), decided 1987–1993, 193 of them Approved. The text says nothing
  about the proposal. Brent has the same placeholder (227 records,
  `semantic/brent/description.yaml`), so LDMS is not Lewisham-only; what it stands for
  is not known.
- **3 under 10 characters:**
  - "Test Test": TEST/22/022223, a test record (dropped).
  - "V": DC/22/129197, Approved 20/01/2023 at 31 Hatcham Park Road SE14. A real
    application with no usable description.
  - "FOOB": DC/21/122666, Closed 20/07/2021 at **120 Fleet Street EC4A** (City of London).
    Likely a consultation response; not confirmed, so it stays loaded.